# Практика 20. Лінійне програмування

**Студент:** Коць Артем  
**Група:** ІТ-42  
**Варіант:** 9  
**Місто:** Суми

## Мета роботи

Сформулювати задачу лінійного програмування для виробничої
дільниці, розв'язати її за допомогою `scipy.optimize.linprog`,
інтерпретувати оптимальний план виробництва, визначити активні
та неактивні ресурсні обмеження і дослідити вплив зміни ресурсів
на максимальний прибуток.

In [1]:
import numpy as np
import pandas as pd

from scipy.optimize import linprog

## Завдання 1. Математична постановка задачі

Нехай:

- `x1` — кількість виробів А, що виробляються за тиждень;
- `x2` — кількість виробів Б, що виробляються за тиждень.

### Цільова функція

Потрібно максимізувати тижневий прибуток:

**max Z = 46*x1 + 60*x2**

### Обмеження по сировині

3*x1 + 2*x2 <= 145

### Обмеження по робочому часу

2*x1 + 5*x2 <= 225

### Обмеження по електроенергії

2*x1 + x2 <= 105

### Умова невід'ємності

x1 >= 0  
x2 >= 0

Отже, потрібно знайти такі кількості виробів А та Б, які
максимізують прибуток і одночасно задовольняють усі три
ресурсні обмеження.

In [2]:
# Прибуток потрібно максимізувати,
# тому для linprog беремо від'ємні коефіцієнти.
c = [-46, -60]

# Обмеження мають форму A_ub @ x <= b_ub
A_ub = [
    [3, 2],   # сировина
    [2, 5],   # робочий час
    [2, 1],   # електроенергія
]

b_ub = [
    145,      # ліміт сировини
    225,      # ліміт часу
    105       # ліміт електроенергії
]

res = linprog(
    c,
    A_ub=A_ub,
    b_ub=b_ub,
    bounds=[
        (0, None),
        (0, None)
    ],
    method="highs"
)

print("res.x =", res.x)
print("res.fun =", res.fun)
print("Максимальний прибуток =", -res.fun)
print("res.status =", res.status)
print("res.message =", res.message)

res.x = [25. 35.]
res.fun = -3250.0
Максимальний прибуток = 3250.0
res.status = 0
res.message = Optimization terminated successfully. (HiGHS Status 7: Optimal)


## Завдання 2. Оптимальний план виробництва

Отриманий розв'язок `res.x` показує оптимальні обсяги виробництва
виробів А та Б.

Максимальний прибуток визначається як `-res.fun`, оскільки в
`linprog` цільову функцію прибутку було подано з протилежним знаком.

Перевіримо також використання кожного ресурсу.

In [3]:
x1, x2 = res.x

raw_used = 3 * x1 + 2 * x2
time_used = 2 * x1 + 5 * x2
electricity_used = 2 * x1 + x2

print(f"Виріб А: {x1:.2f} од.")
print(f"Виріб Б: {x2:.2f} од.")
print(f"Максимальний прибуток: {-res.fun:.2f} грн")
print()
print(f"Сировина: {raw_used:.2f} / 145 кг")
print(f"Час: {time_used:.2f} / 225 год")
print(f"Електроенергія: {electricity_used:.2f} / 105 кВт·год")

Виріб А: 25.00 од.
Виріб Б: 35.00 од.
Максимальний прибуток: 3250.00 грн

Сировина: 145.00 / 145 кг
Час: 225.00 / 225 год
Електроенергія: 85.00 / 105 кВт·год


## Завдання 3. Активні та неактивні ресурсні обмеження

Активним є обмеження, для якого весь доступний ресурс використано,
тобто запас ресурсу дорівнює нулю.

Неактивним є обмеження, для якого після виробництва залишається
невикористаний запас.

In [4]:
# Запаси ресурсів після оптимального плану
slack = res.slack

print(f"Невикористана сировина: {slack[0]:.2f} кг")
print(f"Невикористаний час: {slack[1]:.2f} год")
print(f"Невикористана електроенергія: {slack[2]:.2f} кВт·год")

print("\nАктивні обмеження:")
if np.isclose(slack[0], 0):
    print("- сировина")
if np.isclose(slack[1], 0):
    print("- робочий час")

print("\nНеактивні обмеження:")
if not np.isclose(slack[2], 0):
    print("- електроенергія")

Невикористана сировина: 0.00 кг
Невикористаний час: 0.00 год
Невикористана електроенергія: 20.00 кВт·год

Активні обмеження:
- сировина
- робочий час

Неактивні обмеження:
- електроенергія


## Завдання 4. Зміна активного ресурсу

Сировина є активним ресурсом, тому перевіримо, як зміниться
оптимальний план і максимальний прибуток при збільшенні її запасу
на 15%.

Новий ліміт сировини:

145 * 1.15 = 166.75 кг

In [5]:
# Збільшуємо ліміт сировини на 15%
new_raw_limit = 145 * 1.15

b_ub_active = [
    new_raw_limit,
    225,
    105
]

res_active = linprog(
    c,
    A_ub=A_ub,
    b_ub=b_ub_active,
    bounds=[
        (0, None),
        (0, None)
    ],
    method="highs"
)

print("Новий ліміт сировини:", new_raw_limit, "кг")
print(f"Виріб А: {res_active.x[0]:.2f} од.")
print(f"Виріб Б: {res_active.x[1]:.2f} од.")
print(f"Новий максимальний прибуток: {-res_active.fun:.2f} грн")
print()
print("Нові запаси ресурсів:")
print(f"Сировина: {res_active.slack[0]:.2f} кг")
print(f"Час: {res_active.slack[1]:.2f} год")
print(f"Електроенергія: {res_active.slack[2]:.2f} кВт·год")

Новий ліміт сировини: 166.75 кг
Виріб А: 34.89 од.
Виріб Б: 31.05 од.
Новий максимальний прибуток: 3467.50 грн

Нові запаси ресурсів:
Сировина: 0.00 кг
Час: 0.00 год
Електроенергія: 4.18 кВт·год


## Завдання 5. Зміна неактивного ресурсу

Електроенергія у початковому оптимальному плані використовується
не повністю: залишається 20 кВт·год.

Збільшимо доступний обсяг електроенергії на 50%:

105 * 1.5 = 157.5 кВт·год

Перевіримо, чи зміниться оптимальний план і максимальний прибуток.

In [6]:
# Збільшуємо неактивний ресурс електроенергії на 50%
new_electricity_limit = 105 * 1.5

b_ub_inactive = [
    145,
    225,
    new_electricity_limit
]

res_inactive = linprog(
    c,
    A_ub=A_ub,
    b_ub=b_ub_inactive,
    bounds=[
        (0, None),
        (0, None)
    ],
    method="highs"
)

print("Новий ліміт електроенергії:", new_electricity_limit, "кВт·год")
print(f"Виріб А: {res_inactive.x[0]:.2f} од.")
print(f"Виріб Б: {res_inactive.x[1]:.2f} од.")
print(f"Новий максимальний прибуток: {-res_inactive.fun:.2f} грн")
print()
print("Нові запаси ресурсів:")
print(f"Сировина: {res_inactive.slack[0]:.2f} кг")
print(f"Час: {res_inactive.slack[1]:.2f} год")
print(f"Електроенергія: {res_inactive.slack[2]:.2f} кВт·год")

Новий ліміт електроенергії: 157.5 кВт·год
Виріб А: 25.00 од.
Виріб Б: 35.00 од.
Новий максимальний прибуток: 3250.00 грн

Нові запаси ресурсів:
Сировина: 0.00 кг
Час: 0.00 год
Електроенергія: 72.50 кВт·год


## Порівняння результатів What-if аналізу

Порівняємо початковий оптимальний план із результатами зміни
активного та неактивного ресурсів.

Початковий план:
- виріб А — 25 од.;
- виріб Б — 35 од.;
- прибуток — 3250 грн.

При збільшенні сировини на 15% прибуток збільшується до 3467.50 грн.

При збільшенні електроенергії на 50% оптимальний план та прибуток
не змінюються.

In [7]:
comparison = pd.DataFrame({
    "Сценарій": [
        "Початковий",
        "Сировина +15%",
        "Електроенергія +50%"
    ],
    "Виріб А": [
        res.x[0],
        res_active.x[0],
        res_inactive.x[0]
    ],
    "Виріб Б": [
        res.x[1],
        res_active.x[1],
        res_inactive.x[1]
    ],
    "Прибуток, грн": [
        -res.fun,
        -res_active.fun,
        -res_inactive.fun
    ]
})

comparison.round(2)

,Сценарій,Виріб А,Виріб Б,"Прибуток, грн"
0,Початковий,25.00,35.00,3250.0
1,Сировина +15%,34.89,31.05,3467.5
2,Електроенергія +50%,25.00,35.00,3250.0


## Перевірка коректності оптимального розв'язку

Перевіримо, що отриманий початковий план задовольняє всі ресурсні
обмеження та умови невід'ємності.

In [8]:
x = res.x

checks = {
    "Сировина <= 145": 3 * x[0] + 2 * x[1] <= 145 + 1e-9,
    "Час <= 225": 2 * x[0] + 5 * x[1] <= 225 + 1e-9,
    "Електроенергія <= 105": 2 * x[0] + x[1] <= 105 + 1e-9,
    "x1 >= 0": x[0] >= 0,
    "x2 >= 0": x[1] >= 0,
    "Оптимізація успішна": res.success
}

for name, result in checks.items():
    print(f"{name}: {'OK' if result else 'ПОМИЛКА'}")

print("\nУсі перевірки:", all(checks.values()))

Сировина <= 145: OK
Час <= 225: OK
Електроенергія <= 105: OK
x1 >= 0: OK
x2 >= 0: OK
Оптимізація успішна: OK

Усі перевірки: True


## Контрольні питання

### 1. Що таке задача лінійного програмування?

Задача лінійного програмування — це задача оптимізації лінійної
цільової функції за наявності лінійних обмежень.

### 2. Чому для `linprog` прибуток задається з від'ємним знаком?

`scipy.optimize.linprog` виконує мінімізацію. Тому для максимізації
прибутку функцію прибутку множимо на -1.

### 3. Що означає активне обмеження?

Активне обмеження повністю використовується в оптимальному плані,
тому його запас дорівнює нулю.

### 4. Які обмеження є активними у нашій задачі?

Активними є обмеження по сировині та робочому часу.

### 5. Яке обмеження є неактивним?

Неактивним є обмеження по електроенергії. У початковому плані
залишається 20 кВт·год невикористаної електроенергії.

### 6. Що відбулося після збільшення сировини на 15%?

Оптимальний план змінився, а максимальний прибуток збільшився
з 3250 грн до 3467.50 грн.

### 7. Що відбулося після збільшення електроенергії на 50%?

Оптимальний план і максимальний прибуток не змінилися, оскільки
електроенергія не була вузьким місцем задачі.

### 8. Що таке вузьке місце виробництва?

Вузьке місце — це ресурсне обмеження, яке стримує збільшення
виробництва та використовується повністю.

# Висновок

У практичній роботі було сформульовано та розв'язано задачу
лінійного програмування для виробничої дільниці міста Суми.

Для оптимізації було використано `scipy.optimize.linprog` з методом
`highs`. Оптимальний початковий план становить 25 одиниць виробу А
та 35 одиниць виробу Б. Максимальний тижневий прибуток становить
3250 грн.

У початковому оптимальному плані повністю використовуються сировина
та робочий час, тому вони є активними обмеженнями та основними
вузькими місцями виробництва. Електроенергія використовується не
повністю, тому її обмеження є неактивним.

Збільшення запасу сировини на 15% призвело до зміни оптимального
плану та збільшення прибутку до 3467.50 грн. Натомість збільшення
ліміту електроенергії на 50% не змінило оптимальний план і прибуток.

Отже, для подальшого збільшення прибутку доцільніше збільшувати
ресурси, які є активними обмеженнями, насамперед сировину та
робочий час.